# 05 — Comparação exploratória: regras de risco × modelo preditivo

Este notebook acrescenta uma comparação supervisionada ao Projeto 05. Usa a base sintética PaySim e mantém as mesmas partições cronológicas do baseline; não representa desempenho em produção. **As células ainda não foram executadas neste ambiente** porque o CSV PaySim não está disponível aqui. Execute localmente e interprete os resultados somente depois de revisar os outputs.

O modelo é uma regressão logística com peso balanceado para a classe rara. A lista de variáveis é explícita e exclui o rótulo, sinal pré-existente, identificadores e saldos posteriores à transação. O limiar probabilístico é escolhido apenas na validação para aproximar o volume de alertas do baseline com score ≥ 2; depois é mantido fixo no teste exploratório.

> **Cuidado metodológico:** a EDA já examinou rótulos de todos os períodos. Portanto, mesmo a partição final continua sendo um backtest exploratório, não uma avaliação cega/independente. O modelo não deve ser usado para decisões reais.

## 1. Carregamento e divisão temporal

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.data.prepare_data import find_csv
from src.analysis.risk_rules import chronological_masks, fit_rule_thresholds, score_rules, alert_metrics

csv_path = find_csv(ROOT / 'data' / 'raw')
df = pd.read_csv(csv_path)
train_mask, valid_mask, test_mask = chronological_masks(df)
train = df.loc[train_mask].copy()
valid = df.loc[valid_mask].copy()
test = df.loc[test_mask].copy()
print(f'Treino: {len(train):,} | validação: {len(valid):,} | teste exploratório: {len(test):,}')
display(pd.DataFrame([{'partição': name, 'transações': len(part), 'fraudes': int(part.isFraud.sum()), 'prevalência_pct': 100*part.isFraud.mean(), 'step_inicial': int(part.step.min()), 'step_final': int(part.step.max())} for name, part in [('treino', train), ('validação', valid), ('teste exploratório', test)]]))

## 2. Variáveis disponíveis no momento da transação

Usamos tipo, valor, índice temporal sintético e saldos anteriores, além de razões derivadas com esses campos. Excluímos `isFraud`, `isFlaggedFraud`, `nameOrig`, `nameDest`, `newbalanceOrig` e `newbalanceDest`. Isso reduz vazamento óbvio, mas **não prova** que todos os campos seriam disponíveis ou estáveis em uma operação real.

In [ ]:
NUMERIC_FEATURES = ['step', 'amount', 'oldbalanceOrg', 'oldbalanceDest', 'amount_to_origin_balance', 'origin_balance_after_amount_ratio', 'amount_to_destination_balance']
CATEGORICAL_FEATURES = ['type']
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

def make_features(frame):
    x = frame[['step', 'type', 'amount', 'oldbalanceOrg', 'oldbalanceDest']].copy()
    x['amount_to_origin_balance'] = x['amount'] / x['oldbalanceOrg'].replace(0, np.nan)
    x['origin_balance_after_amount_ratio'] = (x['oldbalanceOrg'] - x['amount']) / x['oldbalanceOrg'].replace(0, np.nan)
    x['amount_to_destination_balance'] = x['amount'] / x['oldbalanceDest'].replace(0, np.nan)
    x[NUMERIC_FEATURES] = x[NUMERIC_FEATURES].replace([np.inf, -np.inf], np.nan)
    return x[FEATURES]

X_train, y_train = make_features(train), train['isFraud'].astype('int8')
X_valid, y_valid = make_features(valid), valid['isFraud'].astype('int8')
X_test, y_test = make_features(test), test['isFraud'].astype('int8')
print('Variáveis usadas:', FEATURES)

## 3. Baseline de regras e modelo logístico

In [ ]:
# O P95 é aprendido apenas no treino. O score não usa rótulos nem campos posteriores.
p95_by_type = fit_rule_thresholds(train)
rule_valid = score_rules(valid, p95_by_type)
rule_test = score_rules(test, p95_by_type)

numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocess = ColumnTransformer([('numeric', numeric_pipe, NUMERIC_FEATURES), ('categorical', categorical_pipe, CATEGORICAL_FEATURES)])
model = Pipeline([('preprocess', preprocess), ('classifier', LogisticRegression(class_weight='balanced', solver='lbfgs', max_iter=300, random_state=2026))])
model.fit(X_train, y_train)
prob_valid = model.predict_proba(X_valid)[:, 1]
prob_test = model.predict_proba(X_test)[:, 1]
print('Modelo ajustado. Não interpretar os scores como probabilidades calibradas: class_weight=balanced altera a prevalência efetiva do ajuste.')

## 4. Limiar de comparação por carga de alertas

Para evitar escolher um limiar sem hipótese operacional, definimos um orçamento **comparativo**: o número de alertas que as regras geraram na validação com score ≥ 2. A regressão recebe um limiar calculado somente na validação para aproximar esse volume. O mesmo limiar numérico é aplicado ao teste; a quantidade no teste pode mudar. Isso compara os métodos sob carga de referência, mas não declara a carga aceitável para uma operação.

In [ ]:
RULE_THRESHOLD = 2
target_alerts_valid = int((rule_valid >= RULE_THRESHOLD).sum())
if target_alerts_valid == 0:
    raise ValueError('O baseline não gerou alertas na validação; não há orçamento comparativo para selecionar o limiar.')
target_alert_rate_valid = target_alerts_valid / len(valid)
model_threshold = float(np.quantile(prob_valid, 1 - target_alert_rate_valid, method='higher'))
print(f'Alertas das regras na validação: {target_alerts_valid:,} ({target_alert_rate_valid:.3%})')
print(f'Limiar do modelo escolhido na validação: {model_threshold:.8f}')
print(f'Alertas do modelo na validação com >= limiar: {(prob_valid >= model_threshold).sum():,}')

## 5. Métricas retrospectivas e comparação

In [ ]:
def evaluate_row(partition, y, rule_score, probability, model_cutoff):
    rule_pred = np.asarray(rule_score) >= RULE_THRESHOLD
    model_pred = np.asarray(probability) >= model_cutoff
    rows = []
    for method, pred, ranking_score in [('Regras (score ≥ 2)', rule_pred, np.asarray(rule_score)), ('Regressão logística', model_pred, np.asarray(probability))]:
        rows.append({'partição': partition, 'método': method, 'alertas': int(pred.sum()), 'taxa_alertas_pct': 100*pred.mean(), 'precisão_pct': 100*precision_score(y, pred, zero_division=0), 'recall_pct': 100*recall_score(y, pred, zero_division=0), 'F1_pct': 100*f1_score(y, pred, zero_division=0), 'average_precision': average_precision_score(y, ranking_score), 'falsos_positivos': int(((pred) & (np.asarray(y)==0)).sum()), 'falsos_negativos': int(((~pred) & (np.asarray(y)==1)).sum())})
    return rows

comparison = pd.DataFrame(
    evaluate_row('Validação', y_valid, rule_valid, prob_valid, model_threshold) +
    evaluate_row('Teste exploratório', y_test, rule_test, prob_test, model_threshold)
)
display(comparison.round(4))

# Matrizes de confusão para transparência sobre falsos positivos/falsos negativos.
for label, y, rule_score, probability in [('Validação', y_valid, rule_valid, prob_valid), ('Teste exploratório', y_test, rule_test, prob_test)]:
    print(f'\n{label} — linhas: real 0/1; colunas: previsto 0/1')
    display(pd.DataFrame({'regras': confusion_matrix(y, np.asarray(rule_score) >= RULE_THRESHOLD).ravel(), 'regressão_logística': confusion_matrix(y, np.asarray(probability) >= model_threshold).ravel()}, index=['TN','FP','FN','TP']))

## 6. Leitura responsável

- Compare a precisão/recall **junto com o volume de alertas** e a AP; nenhuma métrica isolada escolhe a política.
- A prevalência muda entre as janelas temporais, então precisão e carga podem mudar mesmo mantendo o limiar.
- A regressão logística não é automaticamente superior às regras; qualquer diferença é apenas observada neste backtest sintético e exploratório.
- Não use o teste para ajustar variáveis, hiperparâmetros ou limiar. Se houver qualquer decisão posterior baseada nos outputs do teste, ele deixa de ser uma avaliação final intocada.
- Para um resultado confirmatório, obtenha/defina uma janela futura realmente não examinada e congele previamente variáveis, modelo, limiar e métricas.
- Não estimar perdas evitadas, impacto financeiro nem eficácia operacional a partir deste notebook. A fila simulada permanece no notebook 04 e seus KPIs são premissas de cenário, não observações.